# ClipGrab AI Enhance (free)
Upscales a video clip with **Real-ESRGAN** on Google's free GPU and can raise it to **60 fps**.

**Steps:** 1) `Runtime → Change runtime type → T4 GPU` 2) run the cells top to bottom (▶) 3) upload your clip when asked 4) download the result.

Notes: short clips (5–30 s) work best. The free GPU is limited and can disconnect. AI upscaling adds *plausible* detail, it can't recover what was never recorded.

In [ ]:
#@title 1. Check the GPU
import subprocess
print(subprocess.run(["nvidia-smi","-L"],capture_output=True,text=True).stdout or "NO GPU - set Runtime > Change runtime type > T4 GPU, then run again")

In [ ]:
#@title 2. Install (about 2-3 minutes)
%cd /content
!git clone -q https://github.com/xinntao/Real-ESRGAN.git
%cd /content/Real-ESRGAN
!pip -q install basicsr facexlib gfpgan ffmpeg-python
!pip -q install -r requirements.txt
!python setup.py develop -q
# basicsr still imports a torchvision module that newer versions removed - patch it
import glob, re
for f in glob.glob("/usr/local/lib/python3*/dist-packages/basicsr/data/degradations.py"):
    s = open(f).read()
    s = s.replace("from torchvision.transforms.functional_tensor import rgb_to_grayscale", "from torchvision.transforms.functional import rgb_to_grayscale")
    open(f, "w").write(s)
print("installed")

In [ ]:
#@title 3. Upload your video
import os, shutil
from google.colab import files
os.makedirs("/content/in", exist_ok=True)
up = files.upload()
src = list(up.keys())[0]
shutil.move(src, "/content/in/clip.mp4")
print("got", src)

In [ ]:
#@title 4. Settings
target_height = 1080  #@param [720, 1080, 1440, 2160]
model = "realesr-general-x4v3"  #@param ["realesr-general-x4v3", "RealESRGAN_x4plus", "realesr-animevideo"]
make_60fps = True  #@param {type:"boolean"}
quality_crf = 14  #@param {type:"slider", min:10, max:23, step:1}

In [ ]:
#@title 5. Run the upscale
import json, subprocess
info = json.loads(subprocess.run(["ffprobe","-v","error","-select_streams","v:0","-show_entries","stream=width,height","-of","json","/content/in/clip.mp4"],capture_output=True,text=True).stdout)["streams"][0]
w, h = info["width"], info["height"]
short = min(w, h)
scale = round(target_height / short, 3)
print(f"input {w}x{h} -> scale x{scale} (target short side {target_height}px)")
assert scale > 1, "The clip is already at or above that size - pick a higher target."
%cd /content/Real-ESRGAN
!rm -rf /content/out && python inference_realesrgan_video.py -i /content/in/clip.mp4 -n $model -s $scale --suffix up -o /content/out

In [ ]:
#@title 6. Make the final file (fps + bitrate)
import glob, subprocess
up_file = glob.glob("/content/out/*up*.mp4")[0]
vf = "minterpolate=fps=60:mi_mode=mci:mc_mode=aobmc:vsbmc=1" if make_60fps else "null"
cmd = ["ffmpeg","-y","-i",up_file,"-i","/content/in/clip.mp4","-map","0:v:0","-map","1:a:0?","-vf",vf,
       "-c:v","libx264","-preset","slow","-crf",str(quality_crf),"-pix_fmt","yuv420p","-c:a","aac","-b:a","192k","-movflags","+faststart","/content/ClipGrab_enhanced.mp4"]
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stderr[-600:] if r.returncode else "done")
import os; print(round(os.path.getsize("/content/ClipGrab_enhanced.mp4")/1e6,1), "MB")

In [ ]:
#@title 7. Download the result
from google.colab import files
files.download("/content/ClipGrab_enhanced.mp4")